In [3]:
import pandas as pd
import networkx as nx

files_dir = 'C:/Users/hnielsen/Desktop/Genomics'

# Load your final filtered network (adjust filename/threshold to whatever you settled on)
df = pd.read_csv(f'{files_dir}/SparCC/files/sparcc_fdr_significant_pairs_r_filt.csv')

# Option 1 (recommended): positive correlations only
df_positive = df[df['correlation'] > 0].copy()

# Build the graph
G = nx.Graph()
for _, row in df_positive.iterrows():
    G.add_edge(row['ASV_1'], row['ASV_2'], weight=row['correlation'])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Run Louvain community detection
communities = nx.community.louvain_communities(G, weight='weight', seed=42)
print(f'Number of communities found: {len(communities)}')

# Compute modularity score (higher = more well-defined community structure, typically 0.3+ is considered meaningful)
modularity = nx.community.modularity(G, communities, weight='weight')
print(f'Modularity: {modularity:.4f}')

# Build a results table: one row per ASV, with its assigned community and degree
asv_to_community = {}
for comm_id, comm_members in enumerate(communities):
    for asv in comm_members:
        asv_to_community[asv] = comm_id

degree_dict = dict(G.degree())

results = pd.DataFrame({
    'ASV': list(asv_to_community.keys()),
    'community': list(asv_to_community.values())
})
results['degree'] = results['ASV'].map(degree_dict)
results = results.sort_values(['community', 'degree'], ascending=[True, False])

results.to_csv(f'{files_dir}/Louvain/asv_community_assignments.csv', index=False)
print(f'Saved asv_community_assignments.csv')
print()
print('Community sizes:')
print(results['community'].value_counts().sort_index())

Graph: 165 nodes, 854 edges
Number of communities found: 9
Modularity: 0.5798
Saved asv_community_assignments.csv

Community sizes:
community
0     4
1    31
2    20
3    50
4     5
5     9
6    42
7     2
8     2
Name: count, dtype: int64
